A Game Of Thrones

In [3]:
from zipfile import ZipFile

zip_path = '/content/archive (3).zip'

extract_path = '/content/archive'

# Unzip
with ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Unzipped successfully to:", extract_path)

✅ Unzipped successfully to: /content/archive


In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 92.7 MB/s eta 0:00:00


In [2]:
import os
import nltk
from nltk import sent_tokenize
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
import plotly.express as px

In [4]:
book_files = [
    "/content/archive/1 - A Game of Thrones.txt",
    "/content/archive/2 - A Clash of Kings.txt",
    "/content/archive/3 - A Storm of Swords.txt",
    "/content/archive/4 - A Feast for Crows.txt",
    "/content/archive/5 - A Dance with Dragons.txt"
]


In [5]:
nltk.download('punkt_tab')
models={}

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [17]:
for book_path in book_files:
    assert os.path.exists(book_path), f"File not found: {book_path}"
    book_name = os.path.basename(book_path).replace(".txt", "")
    print(f"\nProcessing book: {book_name}")

    encodings_to_try = ['utf-8', 'latin-1', 'cp1252']
    book_text = None
    for encoding in encodings_to_try:
        try:
            with open(book_path, "r", encoding=encoding) as f:
                book_text = f.read()
            print(f"Successfully read with encoding: {encoding}")
            break
        except UnicodeDecodeError:
            print(f"Failed to read with encoding: {encoding}")
            continue

    if book_text is None:
        print(f"Could not read file {book_path} with any of the attempted encodings.")
        continue

    print(f"First 300 characters:\n{book_text[:300]}\n")
    print(f"Total characters in file: {len(book_text)}")

    sentences_raw = sent_tokenize(book_text)
    tokenized_lines = [simple_preprocess(line) for line in sentences_raw]

    print(f"Total sentences detected: {len(tokenized_lines)}")
    print("Example tokens:", tokenized_lines[0][:15])

    got_vec = Word2Vec(
        vector_size=100,
        window=10,
        min_count=2,
        workers=4
    )

    got_vec.build_vocab(tokenized_lines)
    got_vec.train(tokenized_lines, total_examples=got_vec.corpus_count, epochs=got_vec.epochs)

    models[book_name] = got_vec

    print(f"\nTop words similar to 'king' in {book_name}:")
    print(got_vec.wv.most_similar('king', topn=5))

    print(f"\nTop words similar to 'queen' in {book_name}:")
    print(got_vec.wv.most_similar('queen', topn=5))

    print(f"\nSimilarity between 'winter' and 'snow' in {book_name}: {got_vec.wv.similarity('winter', 'snow'):.3f}") #'maelstrom', 'feast',

    print("-" * 100)


Processing book: 1 - A Game of Thrones
Successfully read with encoding: utf-8
First 300 characters:
A Game Of Thrones 
Book One of A Song of Ice and Fire 
By George R. R. Martin 
PROLOGUE 
"We should start back," Gared urged as the woods began to grow dark around them. "The wildlings are 
dead." 
"Do the dead frighten you?" Ser Waymar Royce asked with just the hint of a smile. 
Gared did not rise 

Total characters in file: 1607894
Total sentences detected: 27244
Example tokens: ['game', 'of', 'thrones', 'book', 'one', 'of', 'song', 'of', 'ice', 'and', 'fire', 'by', 'george', 'martin', 'prologue']

Top words similar to 'king' in 1 - A Game of Thrones:
[('name', 0.8794375658035278), ('eddard', 0.8749252557754517), ('lord', 0.8703335523605347), ('place', 0.8642709851264954), ('beloved', 0.8631317019462585)]

Top words similar to 'queen' in 1 - A Game of Thrones:
[('victory', 0.9826722741127014), ('name', 0.9806618690490723), ('daughter', 0.9800238609313965), ('wed', 0.9797865152359009),

In [7]:
first_book = list(models.keys())[2]
first_model = models[first_book]

vectors = first_model.wv.get_normed_vectors()
words = first_model.wv.index_to_key
print(f"\nEmbedding shape for {first_book}: {vectors.shape}")

pca_reducer = PCA(n_components=3)
reduced_vecs = pca_reducer.fit_transform(vectors)

fig = px.scatter_3d(
    x=reduced_vecs[:500, 0],
    y=reduced_vecs[:500, 1],
    z=reduced_vecs[:500, 2],
    color=words[:500],
    title=f"3D PCA Visualization of Word Embeddings ({first_book})"
)
fig.show()


Embedding shape for 3 - A Storm of Swords: (9475, 100)


In [8]:
models["1 - A Game of Thrones"].wv.most_similar("ned")



[('catelyn', 0.9709823727607727),
 ('robb', 0.967660129070282),
 ('jon', 0.9607348442077637),
 ('tyrion', 0.9292322993278503),
 ('he', 0.9120573997497559),
 ('viserys', 0.9061768651008606),
 ('bran', 0.9014723896980286),
 ('joffrey', 0.9006487727165222),
 ('weakling', 0.8957908749580383),
 ('cersei', 0.8932863473892212)]

In [9]:
models["5 - A Dance with Dragons"].wv.most_similar("theon")

[('lump', 0.9764836430549622),
 ('bran', 0.9712318181991577),
 ('dwarf', 0.9661985039710999),
 ('asha', 0.962689220905304),
 ('varamyr', 0.9612817764282227),
 ('griff', 0.9557687044143677),
 ('hazzea', 0.951876699924469),
 ('haggon', 0.951812207698822),
 ('somehow', 0.9505603909492493),
 ('tyrion', 0.949792206287384)]